# 03 - Pairwise judging with a Kaggle-hosted model

**Create this notebook from https://www.kaggle.com/benchmarks/tasks/new** so that `kaggle_benchmarks` and the Kaggle
model proxy (`MODEL_PROXY_URL`, `MODEL_PROXY_API_KEY`) are available, then bring these cells into it
(File -> Import Notebook, or copy and paste). No GPU is needed.
**Add Input -> Notebooks ->** your saved *02 - Generation* notebook version.

Nothing is generated here. The notebook reads saved pairwise JSONL files, judges 5 pairs per request, waits at least
5 seconds between all request attempts, tries a failing batch up to 3 times, and appends results after every batch.
Rerun any cell to continue where it stopped.

## 1. Get the code (taken from the attached generation output; otherwise from GitHub)

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

# ---- EDIT THESE ---------------------------------------------------------------------------
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"   # your GitHub repository
REPO_BRANCH = "main"
CODE_DATASET_DIR = ""   # alternative to GitHub: an uploaded copy, e.g. "/kaggle/input/my-pa2-code/genaiPA2"
# -------------------------------------------------------------------------------------------

WORK = Path("/kaggle/working")
REPO_DIR = WORK / "small-model-alignment"

def find_code_in_inputs():
    """A copy of the repository inside an attached input (an uploaded dataset or an earlier notebook's output)."""
    for marker in sorted(Path("/kaggle/input").glob("**/scripts/judge_pairwise_kaggle.py")):
        return marker.parents[1]
    return None

if REPO_DIR.exists():
    print("repository already present:", REPO_DIR)
else:
    source = Path(CODE_DATASET_DIR) if CODE_DATASET_DIR else None
    if source is None and "YOUR_USERNAME" in REPO_URL:
        source = find_code_in_inputs()
        if source is None:
            raise SystemExit("Set REPO_URL to your GitHub repository, or set CODE_DATASET_DIR, or attach an input "
                             "that contains the code.")
    if source is not None:
        shutil.copytree(source, REPO_DIR, ignore=shutil.ignore_patterns("outputs", ".git", ".venv", "__pycache__"))
        print("copied code from", source)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
os.environ["PYTHONUNBUFFERED"] = "1"   # show training / generation logs live in "!python ..." cells
sys.path.insert(0, str(REPO_DIR))
print("working directory:", os.getcwd())

In [ ]:
# Judging needs only the standard library plus kaggle_benchmarks (or openai for the proxy fallback).
import importlib
for name in ("kaggle_benchmarks", "openai"):
    try:
        module = importlib.import_module(name)
        print(name, getattr(module, "__version__", "installed"))
    except Exception as exc:
        print(name, "NOT AVAILABLE:", type(exc).__name__, exc)

## 2. Collect the pairwise files (and earlier judgments when resuming in a new session)

In [ ]:
PAIRS_DIR = Path("outputs/eval/pairs")
JUDGED_DIR = Path("outputs/eval/judged")
PAIRS_DIR.mkdir(parents=True, exist_ok=True)
JUDGED_DIR.mkdir(parents=True, exist_ok=True)

for source in sorted(Path("/kaggle/input").glob("**/outputs/eval/pairs/*_pairs.jsonl")):
    target = PAIRS_DIR / source.name
    if not target.exists():
        shutil.copyfile(source, target)
# Judgments from an earlier version of THIS notebook (attach that version as input to continue it).
for source in sorted(Path("/kaggle/input").glob("**/outputs/eval/judged/*")):
    target = JUDGED_DIR / source.name
    if source.is_file() and not target.exists():
        shutil.copyfile(source, target)

print("pairs:", sorted(path.name for path in PAIRS_DIR.glob("*_pairs.jsonl")))
print("existing judgment files:", sorted(path.name for path in JUDGED_DIR.glob("*_judged.jsonl")))

## 3. Choose the judge model

`google/gemini-3.6-flash` is the model suggested by the assignment. The cell lists what this notebook actually
exposes. If the suggested model is not available (the first check in section 4 fails), set `JUDGE_MODEL` to a
listed model and name it in the report.

In [ ]:
sys.path.insert(0, str(REPO_DIR / "scripts"))
import judge_pairwise_kaggle as judge_cli

JUDGE_MODEL = "google/gemini-3.6-flash"
BACKEND = "auto"   # kaggle_benchmarks first, then the OpenAI-compatible model proxy
# Saved judgments are reused only for the same backend, judge model and batch size. If you change one of them
# after judging has started, the run stops before sending anything. Set this to True to rename the old result
# files to *.stale<N> and judge every pair again with the new settings.
INVALIDATE_INCOMPATIBLE = False
judge_cli.main(["--list_models"])

In [ ]:
def judge(comparison, max_batches=None):
    pairs_path = PAIRS_DIR / f"{comparison}_pairs.jsonl"
    if not pairs_path.exists():
        print("skipping", comparison, "- no pairs file")
        return None
    argv = ["--pairs_path", str(pairs_path),
            "--output_path", str(JUDGED_DIR / f"{comparison}_judged.jsonl"),
            "--judge_model", JUDGE_MODEL, "--backend", BACKEND,
            "--batch_size", "5", "--sleep_seconds", "5", "--max_retries", "3", "--reuse_judgments"]
    if max_batches:
        argv += ["--max_batches", str(max_batches)]
    if INVALIDATE_INCOMPATIBLE:
        argv += ["--invalidate_incompatible"]
    return judge_cli.main(argv)

## 4. First check: one batch of 5 pairs. Read the reasons before spending more calls.

In [ ]:
judge("dpo_uf_vs_sft", max_batches=1)
from alignment.io_utils import read_jsonl
first_check = JUDGED_DIR / "dpo_uf_vs_sft_judged.jsonl"
if first_check.exists():
    for record in read_jsonl(first_check)[:5]:
        print(record["prompt_id"], "| verdict:", record["verdict"], "| winner:", record["winner_model"], "|", record["reason"])

## 5. Full judging

About 60 requests per 300-pair comparison. If a cell stops (quota, disconnect), run it again: finished pairs are
skipped and failed ones are retried.

In [ ]:
judge("dpo_uf_vs_sft")

In [ ]:
judge("dpo_hh_vs_sft")

In [ ]:
judge("persona_vs_base_style")
judge("persona_vs_base_quality")

In [ ]:
# Optional: SFT vs base (own analysis only).
RUN_OPTIONAL = False
if RUN_OPTIONAL:
    judge("sft_vs_base")

## 6. Summaries

In [ ]:
import json
for path in sorted(JUDGED_DIR.glob("*.summary.json")):
    summary = json.loads(path.read_text())
    print(f"{summary['comparison']}: judge={summary['judge_model']} pairs={summary['total_pairs']} "
          f"wins={summary['candidate_wins']} losses={summary['baseline_wins']} ties={summary['ties']} "
          f"unknown={summary['unknown']} failures={summary['failures']} complete={summary['complete']} "
          f"win_rate_excl_ties={summary['candidate_win_rate_excluding_ties']}")

In [ ]:
!python scripts/summarize_results.py --runs --eval_dir outputs/eval --out_dir outputs/report
!python scripts/package_outputs.py --source outputs/eval/judged outputs/report --zip_path /kaggle/working/judged_outputs.zip
print(open("outputs/report/report_tables.md").read())

Save the notebook version with its output to keep `outputs/eval/judged/*`: judgments, `*.failures.jsonl`,
`*.attempts.jsonl` and `*.summary.json`. Unresolved pairs stay listed in the failures file and are counted
separately in every summary.